## Graph Q&A — Prompt Strategies with Few-Shot Examples

Schema:
- **Nodes**: Movie, Person, Genre
- **Relationships**: `(:Person)-[:ACTED_IN]->(:Movie)`, `(:Person)-[:DIRECTED]->(:Movie)`, `(:Movie)-[:IN_GENRE]->(:Genre)`

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
graph.refresh_schema()

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.3-70b-versatile")

## 14 Few-Shot Examples

Each example pairs a natural language question with the correct Cypher query.
These are injected into the prompt to guide the LLM.

In [ ]:
examples = [
    {
        "question": "Who directed the movie Casino?",
        "query": "MATCH (p:Person)-[:DIRECTED]->(m:Movie {{title: 'Casino'}}) RETURN p.name AS director"
    },
    {
        "question": "What movies did Tom Hanks act in?",
        "query": "MATCH (p:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN m.title AS movie, m.released AS year ORDER BY m.released DESC"
    },
    {
        "question": "What are the top 5 highest rated movies?",
        "query": "MATCH (m:Movie) WHERE m.imdbRating IS NOT NULL RETURN m.title AS movie, m.imdbRating AS rating ORDER BY m.imdbRating DESC LIMIT 5"
    },
    {
        "question": "Which movies belong to the Action genre?",
        "query": "MATCH (m:Movie)-[:IN_GENRE]->(g:Genre {{name: 'Action'}}) RETURN m.title AS movie, m.imdbRating AS rating ORDER BY m.imdbRating DESC"
    },
    {
        "question": "Who are the actors in the movie Pulp Fiction?",
        "query": "MATCH (p:Person)-[:ACTED_IN]->(m:Movie {{title: 'Pulp Fiction'}}) RETURN p.name AS actor"
    },
    {
        "question": "How many movies has Martin Scorsese directed?",
        "query": "MATCH (p:Person {{name: 'Martin Scorsese'}})-[:DIRECTED]->(m:Movie) RETURN COUNT(m) AS total_movies"
    },
    {
        "question": "Which directors have made more than 1 movie?",
        "query": "MATCH (p:Person)-[:DIRECTED]->(m:Movie) WITH p, COUNT(m) AS total WHERE total > 1 RETURN p.name AS director, total ORDER BY total DESC"
    },
    {
        "question": "Which actors appeared in movies directed by Martin Scorsese?",
        "query": "MATCH (d:Person {{name: 'Martin Scorsese'}})-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person) RETURN DISTINCT a.name AS actor, m.title AS movie"
    },
    {
        "question": "What genres does the movie Pulp Fiction belong to?",
        "query": "MATCH (m:Movie {{title: 'Pulp Fiction'}})-[:IN_GENRE]->(g:Genre) RETURN g.name AS genre"
    },
    {
        "question": "Which movies have an IMDB rating above 8?",
        "query": "MATCH (m:Movie) WHERE m.imdbRating > 8 RETURN m.title AS movie, m.imdbRating AS rating ORDER BY m.imdbRating DESC"
    },
    {
        "question": "Who acted in Toy Story?",
        "query": "MATCH (p:Person)-[:ACTED_IN]->(m:Movie {{title: 'Toy Story'}}) RETURN p.name"
    },
    {
        "question": "What genre is Pulp Fiction?",
        "query": "MATCH (m:Movie {{title: 'Pulp Fiction'}})-[:IN_GENRE]->(g:Genre) RETURN g.name"
    },
    {
        "question": "Which movies have an IMDB rating above 8.5?",
        "query": "MATCH (m:Movie) WHERE m.imdbRating > 8.5 RETURN m.title, m.imdbRating ORDER BY m.imdbRating DESC"
    },
    {
        "question": "How many movies did Quentin Tarantino direct?",
        "query": "MATCH (p:Person {{name: 'Quentin Tarantino'}})-[:DIRECTED]->(m:Movie) RETURN COUNT(m) AS total"
    }
]

print(f"Loaded {len(examples)} few-shot examples")

## Build Few-Shot Prompt with SemanticSimilarityExampleSelector

Selects the most relevant examples for each incoming question using embeddings.

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,
    FAISS,
    k=3,
    input_keys=["question"]
)

In [ ]:
example_prompt = PromptTemplate(
    input_variables=["question", "query"],
    template="Question: {question}
Cypher: {query}"
)

few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix=(
        "You are a Neo4j Cypher expert. Generate a Cypher query for the given question.
"
        "Use only the schema below. Return only the Cypher query, no explanations.
"
        "Schema: {schema}

"
        "Here are some similar examples:"
    ),
    suffix="Question: {question}
Cypher:",
    input_variables=["schema", "question"]
)

print(few_shot_prompt.format(schema=graph.schema, question="Who directed Titanic?"))

## Wire into GraphCypherQAChain

In [ ]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    cypher_prompt=few_shot_prompt,
    return_intermediate_steps=True,
    allow_dangerous_requests=True
)

## Test Queries

In [ ]:
test_questions = [
    "Who directed the movie Casino?",
    "What movies did Tom Hanks act in?",
    "What are the top 3 movies in the Drama genre?",
    "What genre is Pulp Fiction?",
    "Which actors appeared in movies directed by Martin Scorsese?",
]

for q in test_questions:
    response = chain.invoke({"query": q})
    print(f"Q: {q}")
    print(f"Cypher: {response['intermediate_steps'][0]['query']}")
    print(f"Answer: {response['result']}")
    print("-" * 60)